In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from torchvision.transforms.functional import to_tensor

sys.path.append("..")
from src import *

# Compute Metrics

In [ ]:
dataset = ObjaverseDataset3D()

In [ ]:
TESTSET_DIR = Path("dataset/test")
REND_DIR = Path("renderings")
TEST_DIR = Path("tests")
dirs = [
    "gt",
    "sd15_mlsd",
    "sd15_ours",
    "sdxl_ours",
    "sdxl_mlsd_llite",
    "sdxl_ours_llite",
]
testset = pd.read_json(TESTSET_DIR / "metadata.jsonl", orient="records", lines=True)
testset.index = pd.Series(testset.uv_file_name.map(lambda x: Path(x).stem), name="uid")

In [ ]:
metrics: dict[str, Metric] = {
    "psnr": PSNRMetric(),
    "ssim": SSIMMetric(),
    "lpips": LPIPSMetric(),
    "fid": FIDMetric(),
    "clipiqa": CLIPIQAMetric(),
    "clip": CLIPMetric(),
    "brisque": BRISQUEMetric(),
}

In [ ]:
def path2tensor(path: Path) -> torch.Tensor:
    with Image.open(path) as img:
        img = img.resize((512, 512))
        if img.mode in ("RGBA", "LA"):
            white_bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
            img = Image.alpha_composite(white_bg, img.convert("RGBA"))
        return to_tensor(img.convert("RGB")).unsqueeze(0)

In [ ]:
def compute_metrics(tag, testset):
    views = 3
    uids = testset.index
    y_tex = torch.empty((len(uids), 3, 512, 512))
    gt_tex = torch.empty_like(y_tex)
    y_ren = torch.empty((len(uids), views, 3, 512, 512))
    gt_ren = torch.empty_like(y_ren)
    captions = []

    cprint("yellow:Preprocessing testset...")
    for i, uid in tqdm(enumerate(uids)):
        y_tex[i] = path2tensor(TEST_DIR / tag / f"{uid}.png")
        gt_tex[i] = path2tensor(TESTSET_DIR / "diffuse" / f"{uid}.png")
        for view in range(views):
            y_ren[i, view] = path2tensor(REND_DIR / tag / f"{uid[:-2]}_{view}.png")
            gt_ren[i, view] = path2tensor(REND_DIR / "gt" / f"{uid[:-2]}_{view}.png")
        captions.append(testset.loc[uid].caption)

    cprint(f"yellow:Computing metrics for ({tag})...")
    for k, metric in metrics.items():
        if metric.need_renders:
            m = metric(y_ren, gt_ren, captions)
        else:
            m = metric(y_tex, gt_tex, captions)
        cprint(f"green:{k}", f"blue:{m:.3f}")

In [ ]:
compute_metrics("sd15_mlsd", testset=testset)
compute_metrics("sd15_ours", testset=testset)
compute_metrics("sdxl_ours", testset=testset)
compute_metrics("sdxl_mlsd_llite", testset=testset)
compute_metrics("sdxl_ours_llite", testset=testset)

### Stable Diffusion 1.5

| Model       |  $\text{PSNR} ↑$ |  $\text{SSIM} ↑$ | $\text{LPIPS} ↓$ |   $\text{FID} ↓$   | $\text{CLIP-IQA} ↑$ |  $\text{CLIP} ↑$ | $\text{BRISQUE} ↓$ |
| ----------- | :--------------: | :--------------: | :--------------: | :----------------: | :-----------------: | :--------------: | :----------------: |
| `sd15_mlsd` |      $7.971$     |      $0.244$     |      $0.812$     |      $233.044$     |       $0.826$       |      $0.236$     |  $\mathbf{74.906}$ |
| `sd15_ours` | $\mathbf{8.438}$ | $\mathbf{0.284}$ | $\mathbf{0.789}$ | $\mathbf{229.222}$ |       $\mathbf{0.826}$       | $\mathbf{0.239}$ |      $78.941$      |

### Stable Diffusion XL

| Model             |  $\text{PSNR} ↑$ |  $\text{SSIM} ↑$ | $\text{LPIPS} ↓$ |   $\text{FID} ↓$   | $\text{CLIP-IQA} ↑$ |  $\text{CLIP} ↑$ | $\text{BRISQUE} ↓$ |
| ----------------- | :--------------: | :--------------: | :--------------: | :----------------: | :-----------------: | :--------------: | :----------------: |
| `sdxl_mlsd` |      $8.292$     |      $0.127$     |      $0.906$     |      $300.528$     |       $0.828$       |      $0.177$     |      $80.624$      |
| `sdxl_ours` | $\mathbf{9.050}$ | $\mathbf{0.371}$ | $\mathbf{0.782}$ |      $\mathbf{260.352}$     |       $\mathbf{0.828}$       | $\mathbf{0.242}$ |      $\mathbf{77.507}$      |